# Lesson 1.5 — Decomposition and the Architecture Decision Record

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Decompose a complex problem into independently testable sub-capabilities
- Capture a decision, its alternatives, and consequences in an ADR
- Produce the ADR artifact reused by the capstone

**You will produce:** A written `ADR-001.md` Architecture Decision Record for the pattern choice.

## Concepts

**Decomposition** breaks a hard problem into sub-capabilities that can each be prompted, retrieved for, evaluated, and owned independently — the basis of both workflows and multi-agent designs.

An **ADR (Architecture Decision Record)** is a short, immutable document that captures one significant decision: its *context*, the *options considered*, the *decision*, and the *consequences* (including what you are giving up). ADRs are the architect's audit trail; they let future teams understand *why*, not just *what*. The exam's Domain 6 explicitly tests documenting architectures and communicating trade-offs — the ADR is the canonical vehicle, and you will reference this one throughout the capstone.

> **Reference:** [Anthropic — Documentation home](https://docs.anthropic.com/en/home)

In [ ]:
# !pip install anthropic
# Guarded Claude client setup — every lesson runs with or without an API key.
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic()
        LIVE = True
    else:
        client = None
        LIVE = False
except Exception as e:  # SDK not installed in this kernel
    client = None
    LIVE = False
    print("anthropic SDK unavailable:", e)

# Current Claude model family (confirm exact IDs at https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = {
    "opus": "claude-opus-4-20250514",    # deepest reasoning, highest cost/latency
    "sonnet": "claude-sonnet-4-20250514",# balanced default workhorse
    "haiku": "claude-3-5-haiku-20241022",# fastest/cheapest for simple tasks
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    """Live call when a key exists; otherwise a deterministic mock so notebooks run."""
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system or "You are a helpful solution architect.",
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Decompose the prior-auth problem

We split the end-to-end task into sub-capabilities, each with an owner and a testable success check. This directly feeds pattern and evaluation design.

In [ ]:
subcaps = [
  {"cap": "PHI redaction", "test": "0 PHI leaks on 100-doc gold set"},
  {"cap": "policy retrieval", "test": "top-5 recall >= 0.9 on labeled queries"},
  {"cap": "recommendation drafting", "test": ">=95% agreement vs nurse"},
  {"cap": "citation validation", "test": "every claim maps to a retrieved chunk"},
]
for s in subcaps:
    print(f"- {s['cap']:24} measured by: {s['test']}")

## Code-Along: Step 2 — Write the ADR

We capture the pattern decision (workflow over agentic) as an ADR with explicit alternatives and consequences, then persist it.

In [ ]:
ADR = '''# ADR-001: Use a workflow pattern for prior-auth triage

## Status
Accepted

## Context
High-stakes, auditable healthcare decisions with a p95<20s budget and HIPAA
constraints. A human nurse must approve every final decision.

## Decision
Use a **workflow** (routing + prompt chaining) with code-controlled steps:
redact -> retrieve policy -> draft recommendation -> validate citations -> human review.

## Alternatives considered
- **Augmented LLM (single call):** too little control for multi-step validation.
- **Agentic loop:** flexibility unneeded; hurts auditability, latency, and cost.

## Consequences
+ Deterministic, auditable control flow; each step independently evaluable.
+ Easier to meet SLA and compliance review.
- Less adaptive to novel request shapes; new branches need code changes.
'''
with open("ADR-001.md", "w", encoding="utf-8") as fh:
    fh.write(ADR)
print(ADR)

## Exercise

Write **ADR-002** for a *different* decision in this system — for example choosing the model tier (Sonnet vs Haiku vs Opus) for the drafting step, or choosing self-hosted vs managed vector storage. Use the same four sections and record at least two alternatives and one consequence you are explicitly accepting.

In [ ]:
# TODO: write ADR-002 with Status / Context / Decision / Alternatives / Consequences
adr2 = '''# ADR-002: ...
'''
# TODO: write adr2 to ADR-002.md and print it

## Key Takeaways

- Decompose into sub-capabilities that are independently prompted, retrieved, and evaluated.
- An ADR records context, options, decision, and consequences — the architect's audit trail.
- Documenting what you gave up is as important as what you chose.
- ADRs are tested directly in Domain 6 and anchor the capstone deliverable.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.